In [1]:
import pandas as pd
import numpy as np

In [ ]:
#Geração de dados para modelo de previsão - regressão
#Dados com mais sentido
np.random.seed(42)  # Para reprodutibilidade

# Função para gerar horários realistas
def gerar_horario():
    horas = np.random.choice([8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20], p=[0.12, 0.11, 0.10, 0.08, 0.12, 0.11, 0.10, 0.09, 0.07, 0.05, 0.03, 0.02])
    minutos = np.random.choice([0, 15, 30, 45])
    return f"{horas:02d}:{minutos:02d}"

# Dias da semana com pesos (segunda mais movimentada)
dias = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 'sexta-feira', 'sábado']
dias_pesos = [0.25, 0.20, 0.20, 0.18, 0.15, 0.02]

# Tipos de consulta com tempos médios reais
tipos_consulta = {
    'Clínico Geral': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.30},
    'Pediatra': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.15},
    'Ginecologista': {'tempo_medio': 30, 'variacao': 15, 'peso': 0.12},
    'Ortopedista': {'tempo_medio': 35, 'variacao': 18, 'peso': 0.10},
    'Dermatologista': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.10},
    'Cardiologista': {'tempo_medio': 40, 'variacao': 20, 'peso': 0.08},
    'Neurologista': {'tempo_medio': 45, 'variacao': 22, 'peso': 0.06},
    'Oftalmologista': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.04},
    'Psicólogo': {'tempo_medio': 50, 'variacao': 25, 'peso': 0.03},
    'Nutricionista': {'tempo_medio': 35, 'variacao': 15, 'peso': 0.02}
}

# Lista para armazenar os dados
dados = []

for i in range(100):
    # Seleciona tipo de consulta baseado nos pesos
    tipo = np.random.choice(list(tipos_consulta.keys()), p=[t['peso'] for t in tipos_consulta.values()])
    tempo_medio_tipo = tipos_consulta[tipo]['tempo_medio']
    variacao_tipo = tipos_consulta[tipo]['variacao']

    # Gera features
    horario = gerar_horario()
    hora = int(horario.split(':')[0])

    dia_semana = np.random.choice(dias, p=dias_pesos)

    # Pacientes na fila (varia por horário - mais de manhã e início da tarde)
    if hora < 10 or (14 <= hora < 16):
        pacientes_fila = int(np.random.normal(12, 4))
    elif hora < 12:
        pacientes_fila = int(np.random.normal(8, 3))
    else:
        pacientes_fila = int(np.random.normal(6, 2))
    pacientes_fila = max(1, min(25, pacientes_fila))

    # Pacientes agendados no mesmo horário (0-3 geralmente)
    pacientes_mesmo_horario = np.random.poisson(1.5)
    pacientes_mesmo_horario = min(4, pacientes_mesmo_horario)

    # Hábito de atraso do médico (varia por tipo e dia)
    atraso_base = np.random.normal(8, 5)
    if dia_semana == 'segunda-feira':
        atraso_base += 5
    if tipo in ['Cardiologista', 'Neurologista']:
        atraso_base += 10
    medico_habito_atraso = max(0, min(40, int(atraso_base)))

    # Horário de pico (manhã cedo ou começo da tarde)
    horario_pico = 'sim' if (hora < 10 or (14 <= hora < 16)) else 'não'

    # Cálculo do tempo de espera real (target)
    # Fórmula realista: fila * tempo_medio + atraso_médico + variação
    tempo_espera_base = (pacientes_fila * (tempo_medio_tipo / 60)) * 60  # converte para minutos
    tempo_espera_base += medico_habito_atraso

    # Adiciona variação baseada no tipo
    variacao = np.random.normal(0, variacao_tipo)

    # Efeito do dia da semana
    if dia_semana == 'segunda-feira':
        tempo_espera_base += 10
    elif dia_semana == 'sexta-feira':
        tempo_espera_base -= 5

    # Efeito do horário de pico
    if horario_pico == 'sim':
        tempo_espera_base += 8

    tempo_de_espera = max(5, int(tempo_espera_base + variacao))

    # Cria o registro
    registro = {
        'horario': horario,
        'dia_semana': dia_semana,
        'tipo_consulta': tipo,
        'pacientes_na_fila_antes': pacientes_fila,
        'pacientes_agendados_no_mesmo_horario': pacientes_mesmo_horario,
        'tempo_medio_consulta_tipo_min': tempo_medio_tipo,
        'medico_habito_atraso': medico_habito_atraso,
        'horario_pico': horario_pico,
        'tempo_de_espera_min': tempo_de_espera
    }

    dados.append(registro)

# CONVERSÃO PARA DATAFRAME
df = pd.DataFrame(dados)

# INSERINDO VALORES NULOS ESTRATÉGICOS (aproximadamente 10-15% nulos aleatórios)
colunas_com_nan = ['pacientes_na_fila_antes', 'pacientes_agendados_no_mesmo_horario',
                   'tempo_medio_consulta_tipo_min', 'medico_habito_atraso', 'horario_pico']

for col in colunas_com_nan:
    # Entre 5% e 15% de nulos por coluna
    nulos_idx = np.random.choice(df.index, size=int(len(df) * np.random.uniform(0.05, 0.15)), replace=False)
    df.loc[nulos_idx, col] = np.nan

# Adiciona alguns nulos no horario e dia_semana (raros)
nulos_horario = np.random.choice(df.index, size=3, replace=False)
df.loc[nulos_horario, 'horario'] = np.nan

nulos_dia = np.random.choice(df.index, size=2, replace=False)
df.loc[nulos_dia, 'dia_semana'] = np.nan

# ORDENA POR HORÁRIO (para fazer sentido temporal)
df = df.sort_values('horario').reset_index(drop=True)

# CONVERTE PARA O FORMATO DICIONÁRIO SOLICITADO
bd_tmp_espera = df.to_dict('list')
#df.to_csv('bd_tmp_espera2.csv', index=False)

In [ ]:

#Para gerar arquivo para prever tempo_de_espera_min
def gerar_horario():
    horas = np.random.choice([8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20], p=[0.12, 0.11, 0.10, 0.08, 0.12, 0.11, 0.10, 0.09, 0.07, 0.05, 0.03, 0.02])
    minutos = np.random.choice([0, 15, 30, 45])
    return f"{horas:02d}:{minutos:02d}"

# Dias da semana com pesos (segunda mais movimentada)
dias = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 'sexta-feira', 'sábado']
dias_pesos = [0.25, 0.20, 0.20, 0.18, 0.15, 0.02]

# Tipos de consulta com tempos médios reais
tipos_consulta = {
    'Clínico Geral': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.30},
    'Pediatra': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.15},
    'Ginecologista': {'tempo_medio': 30, 'variacao': 15, 'peso': 0.12},
    'Ortopedista': {'tempo_medio': 35, 'variacao': 18, 'peso': 0.10},
    'Dermatologista': {'tempo_medio': 20, 'variacao': 10, 'peso': 0.10},
    'Cardiologista': {'tempo_medio': 40, 'variacao': 20, 'peso': 0.08},
    'Neurologista': {'tempo_medio': 45, 'variacao': 22, 'peso': 0.06},
    'Oftalmologista': {'tempo_medio': 25, 'variacao': 12, 'peso': 0.04},
    'Psicólogo': {'tempo_medio': 50, 'variacao': 25, 'peso': 0.03},
    'Nutricionista': {'tempo_medio': 35, 'variacao': 15, 'peso': 0.02}
}

# Lista para armazenar os dados
dados = []

for i in range(20):
    # Seleciona tipo de consulta baseado nos pesos
    tipo = np.random.choice(list(tipos_consulta.keys()), p=[t['peso'] for t in tipos_consulta.values()])
    tempo_medio_tipo = tipos_consulta[tipo]['tempo_medio']
    variacao_tipo = tipos_consulta[tipo]['variacao']

    # Gera features
    horario = gerar_horario()
    hora = int(horario.split(':')[0])

    dia_semana = np.random.choice(dias, p=dias_pesos)

    # Pacientes na fila (varia por horário - mais de manhã e início da tarde)
    if hora < 10 or (14 <= hora < 16):
        pacientes_fila = int(np.random.normal(12, 4))
    elif hora < 12:
        pacientes_fila = int(np.random.normal(8, 3))
    else:
        pacientes_fila = int(np.random.normal(6, 2))
    pacientes_fila = max(1, min(25, pacientes_fila))

    # Pacientes agendados no mesmo horário (0-3 geralmente)
    pacientes_mesmo_horario = np.random.poisson(1.5)
    pacientes_mesmo_horario = min(4, pacientes_mesmo_horario)

    # Hábito de atraso do médico (varia por tipo e dia)
    atraso_base = np.random.normal(8, 5)
    if dia_semana == 'segunda-feira':
        atraso_base += 5
    if tipo in ['Cardiologista', 'Neurologista']:
        atraso_base += 10
    medico_habito_atraso = max(0, min(40, int(atraso_base)))

    # Horário de pico (manhã cedo ou começo da tarde)
    horario_pico = 'sim' if (hora < 10 or (14 <= hora < 16)) else 'não'

    # Cria o registro
    registro = {
        'horario': horario,
        'dia_semana': dia_semana,
        'tipo_consulta': tipo,
        'pacientes_na_fila_antes': pacientes_fila,
        'pacientes_agendados_no_mesmo_horario': pacientes_mesmo_horario,
        'tempo_medio_consulta_tipo_min': tempo_medio_tipo,
        'medico_habito_atraso': medico_habito_atraso,
        'horario_pico': horario_pico
    }

    dados.append(registro)


df_to_predict = pd.DataFrame(dados)
df_to_predict.to_csv('bd_tmp_espera_to_predict.csv', index=False)
df_to_predict.head(5)

In [ ]:
#1 tentativa de gerar dados para o modelo de classificação

rng = np.random.default_rng()
def gerar_horario():
    horas = np.random.choice([8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20], p=[0.12, 0.11, 0.10, 0.08, 0.12, 0.11, 0.10, 0.09, 0.07, 0.05, 0.03, 0.02])
    minutos = np.random.choice([0, 15, 30, 45])
    return f"{horas:02d}:{minutos:02d}"
def gerar_idade():
  idade = rng.integers(low=18, high=90)
  return idade
def gerar_distancia():
  distancia = rng.uniform(1.0, 60.5)
  return distancia
# Dias da semana com pesos (segunda mais movimentada)
dias = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 'sexta-feira', 'sábado']
dias_pesos = [0.25, 0.20, 0.20, 0.18, 0.15, 0.02]

transporte_pesos = [0.20, 0.25, 0.20, 0.12, 0.05, 0.18]

# Tipos de consulta com tempos médios reais
tipos_consulta = {
    'Clínico Geral': {'peso': 0.30},
    'Pediatra': {'peso': 0.15},
    'Ginecologista': {'peso': 0.12},
    'Ortopedista': {'peso': 0.10},
    'Dermatologista': {'peso': 0.10},
    'Cardiologista': {'peso': 0.08},
    'Neurologista': {'peso': 0.06},
    'Oftalmologista': {'peso': 0.04},
    'Psicólogo': {'peso': 0.03},
    'Nutricionista': {'peso': 0.02}
}

# Lista para armazenar os dados
dados = []

for i in range(400):
    # Seleciona tipo de consulta baseado nos pesos

    atrasos = {
        'baixo': {'peso': 0.30},
        'medio': {'peso': 0.40},
        'elevado': {'peso': 0.30}
    }

    tipo = np.random.choice(list(tipos_consulta.keys()), p=[t['peso'] for t in tipos_consulta.values()])

    if(tipo in ['Cardiologista', 'Ginecologista', 'Neurologista']):
      atrasos['baixo']['peso'] = 0.30
      atrasos['medio']['peso'] = 0.35
      atrasos['elevado']['peso'] = 0.35


    # Gera features
    horario = gerar_horario()
    hora = int(horario.split(':')[0])

    dia_semana = np.random.choice(dias, p=dias_pesos)
    transporte = np.random.choice(['carro', 'onibus', 'moto', 'bicleta', 'caminhada', 'trem'], p=transporte_pesos)

    if(transporte in ['onibus', 'caminhada', 'trem']):
      atrasos['baixo']['peso'] = 0.20
      atrasos['medio']['peso'] = 0.40
      atrasos['elevado']['peso'] = 0.40

    horario_pico = 'sim' if (hora < 10 or (14 <= hora < 16)) else 'não'

    if(horario_pico == 'sim'):
      atrasos['baixo']['peso'] = 0.20
      atrasos['medio']['peso'] = 0.30
      atrasos['elevado']['peso'] = 0.50

    opcoes = ['sim', 'não']
    pcd = np.random.choice(opcoes, p=[0.3, 0.7])

    if(pcd == 'sim'):
      atrasos['baixo']['peso'] = 0.20
      atrasos['medio']['peso'] = 0.30
      atrasos['elevado']['peso'] = 0.50

    idade = gerar_idade()

    if(idade < 35):
      atrasos['baixo']['peso'] = 0.20
      atrasos['medio']['peso'] = 0.30
      atrasos['elevado']['peso'] = 0.50

    distancia = gerar_distancia()

    if(distancia > 30):
      atrasos['baixo']['peso'] = 0.20
      atrasos['medio']['peso'] = 0.25
      atrasos['elevado']['peso'] = 0.55

    if(distancia < 15):
      atrasos['baixo']['peso'] = 0.65
      atrasos['medio']['peso'] = 0.30
      atrasos['elevado']['peso'] = 0.05

    if(distancia < 5):
      atrasos['baixo']['peso'] = 0.75
      atrasos['medio']['peso'] = 0.20
      atrasos['elevado']['peso'] = 0.05

    historico_atrasos = np.random.choice(list(atrasos.keys()), p=[t['peso'] for t in atrasos.values()])

    # Cria o registro
    registro = {
        'idade': idade,
        'consulta': tipo,
        'distacia_local_de_partida_e_consultorio_km': f"{distancia:.2f}",
        'tipo_de_transporte': transporte,
        'dia_semana': dia_semana,
        'horario_da_consulta': horario,
        'horario_pico': horario_pico,
        'PcD': pcd,
        'histórico_de_atrasos': historico_atrasos
    }

    dados.append(registro)
data = pd.DataFrame(dados)
data.head(10)

In [ ]:
#2 tentativa para gerar dados para o modelo de classficação
import numpy as np
import pandas as pd

rng = np.random.default_rng()

def gerar_horario():
    horas = np.random.choice([8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20],
                             p=[0.12, 0.11, 0.10, 0.08, 0.12, 0.11, 0.10, 0.09, 0.07, 0.05, 0.03, 0.02])
    minutos = np.random.choice([0, 15, 30, 45])
    return f"{horas:02d}:{minutos:02d}"

def gerar_idade():
    return rng.integers(low=18, high=90)

def gerar_distancia_com_ruido(classe):
    """
    Gera distâncias sobrepostas entre as classes para simular dados reais
    """
    if classe == 'baixo':
        # Centro em 10km, mas pode variar muito
        distancia = rng.normal(loc=10, scale=8)
        distancia = np.clip(distancia, 1, 35)
    elif classe == 'medio':
        # Centro em 25km, com sobreposição com baixo e elevado
        distancia = rng.normal(loc=25, scale=10)
        distancia = np.clip(distancia, 8, 50)
    else:  # elevado
        # Centro em 40km, mas pode ser menor
        distancia = rng.normal(loc=40, scale=12)
        distancia = np.clip(distancia, 15, 70)

    return distancia

def gerar_pcd_com_probabilidade(classe):
    """
    PcD tem influência mas não é determinístico
    """
    if classe == 'baixo':
        prob_pcd = 0.15  # menor chance de ser PcD
    elif classe == 'medio':
        prob_pcd = 0.25
    else:  # elevado
        prob_pcd = 0.35  # maior chance de ser PcD

    return np.random.choice(['sim', 'não'], p=[prob_pcd, 1-prob_pcd])

def gerar_atraso_baseado_em_features(distancia, idade, horario_pico, pcd, transporte):
    """
    Calcula probabilidade de atraso baseada em múltiplas features (realista)
    """
    # Fatores que aumentam chance de atraso
    score = 0

    # Distância (quanto maior, pior)
    if distancia > 30:
        score += 0.4
    elif distancia > 20:
        score += 0.2
    elif distancia < 10:
        score -= 0.1

    # Idade
    if idade > 70:
        score += 0.2
    elif idade < 25:
        score -= 0.1

    # Horário de pico
    if horario_pico == 'sim':
        score += 0.3

    # PcD
    if pcd == 'sim':
        score += 0.2

    # Transporte
    if transporte in ['onibus', 'trem']:
        score += 0.2
    elif transporte == 'caminhada':
        score -= 0.1

    # Adiciona ruído aleatório (importante!)
    score += rng.normal(0, 0.15)

    # Converte score em probabilidades para ['baixo', 'medio', 'elevado']
    # Ensure all raw probabilities are positive and then normalize them.
    # Using softmax-like approach:
    epsilon = 1e-9 # To prevent probabilities from being exactly zero or negative

    # Higher score should lead to higher elevado probability
    # Lower score should lead to higher baixo probability
    # Medio probability should be higher for intermediate scores

    raw_prob_elevado = np.exp(score) + epsilon
    raw_prob_baixo = np.exp(-score) + epsilon # Inversely related to score
    # For medium, make it peak around score 0 and decrease as score goes to extremes
    raw_prob_medio = np.exp(-abs(score) * 0.5) + epsilon # Use a factor like 0.5 to control the spread

    total_raw_probs = raw_prob_elevado + raw_prob_medio + raw_prob_baixo

    prob_atraso_elevado = raw_prob_elevado / total_raw_probs
    prob_atraso_medio = raw_prob_medio / total_raw_probs
    prob_atraso_baixo = raw_prob_baixo / total_raw_probs

    # Escolhe a classe
    return np.random.choice(['baixo', 'medio', 'elevado'],
                           p=[prob_atraso_baixo, prob_atraso_medio, prob_atraso_elevado])

# Configurações
dias = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 'sexta-feira', 'sábado']
dias_pesos = [0.25, 0.20, 0.20, 0.18, 0.15, 0.02]
transporte_pesos = [0.20, 0.25, 0.20, 0.12, 0.05, 0.18]
transportes = ['carro', 'onibus', 'moto', 'bicleta', 'caminhada', 'trem']

tipos_consulta = {
    'Clínico Geral': 0.30,
    'Pediatra': 0.15,
    'Ginecologista': 0.12,
    'Ortopedista': 0.10,
    'Dermatologista': 0.10,
    'Cardiologista': 0.08,
    'Neurologista': 0.06,
    'Oftalmologista': 0.04,
    'Psicólogo': 0.03,
    'Nutricionista': 0.02
}

# Gerar dados
dados = []
n_amostras = 525  # 175 * 3 (mantém quantidade similar)

for i in range(n_amostras):
    # Gera features básicas primeiro
    tipo = np.random.choice(list(tipos_consulta.keys()), p=list(tipos_consulta.values()))
    horario = gerar_horario()
    hora = int(horario.split(':')[0])
    dia_semana = np.random.choice(dias, p=dias_pesos)
    transporte = np.random.choice(transportes, p=transporte_pesos)
    horario_pico = 'sim' if (hora < 10 or (14 <= hora < 16)) else 'não'
    idade = gerar_idade()

    # Gera classe de atraso BASEADA nas features (não o contrário!)
    # Primeiro sorteia uma classe "tendência"
    tendencia_classe = np.random.choice(['baixo', 'medio', 'elevado'], p=[0.33, 0.34, 0.33])

    # Gera distância baseada na tendência (mas com sobreposição)
    distancia = gerar_distancia_com_ruido(tendencia_classe)

    # Gera PcD baseado na tendência
    pcd = gerar_pcd_com_probabilidade(tendencia_classe)

    # Determina o atraso REAL baseado em todas as features (com ruído)
    atraso = gerar_atraso_baseado_em_features(distancia, idade, horario_pico, pcd, transporte)

    # Cria o registro
    registro = {
        'idade': idade,
        'consulta': tipo,
        'distacia_local_de_partida_e_consultorio_km': f"{distancia:.2f}",
        'tipo_de_transporte': transporte,
        'dia_semana': dia_semana,
        'horario_da_consulta': horario,
        'horario_pico': horario_pico,
        'PcD': pcd,
        'histórico_de_atrasos': atraso
    }
    dados.append(registro)

data = pd.DataFrame(dados)

# Embaralha
df_shuffle = data.sample(frac=1).reset_index(drop=True)

# Verifica distribuição
print("Distribuição das classes:")
print(df_shuffle['histórico_de_atrasos'].value_counts())

# Salva
df_shuffle.to_csv('data_to_classifier_realistic.csv', index=False)
